# 1.find and inspect that suspicious blank row:

In [8]:
import pandas as pd
import os

DATA_DIR = "../data/raw"  # adjust if needed

coverage = pd.read_excel(os.path.join(DATA_DIR, "coverage-data.xlsx"))
incidence = pd.read_excel(os.path.join(DATA_DIR, "incidence-rate-data.xlsx"))
cases = pd.read_excel(os.path.join(DATA_DIR, "reported-cases-data.xlsx"))
intro = pd.read_excel(os.path.join(DATA_DIR, "vaccine-introduction-data.xlsx"))
schedule = pd.read_excel(os.path.join(DATA_DIR, "vaccine-schedule-data.xlsx"))

print("Loaded all tables.")

Loaded all tables.


In [9]:
print(incidence[incidence['CODE'].isna()])
print(cases[cases['CODE'].isna()])
print(intro[intro['COUNTRYNAME'].isna()])

                               GROUP CODE NAME  YEAR DISEASE  \
84945  Created: 2025-02-01 16:03 UTC  NaN  NaN   NaN     NaN   

      DISEASE_DESCRIPTION DENOMINATOR  INCIDENCE_RATE  
84945                 NaN         NaN             NaN  
                               GROUP CODE NAME  YEAR DISEASE  \
84869  Created: 2025-02-01 16:02 UTC  NaN  NaN   NaN     NaN   

      DISEASE_DESCRIPTION  CASES  
84869                 NaN    NaN  
                           ISO_3_CODE COUNTRYNAME WHO_REGION  YEAR  \
138320  Created: 2025-02-01 07:09 UTC         NaN        NaN   NaN   

       DESCRIPTION INTRO  
138320         NaN   NaN  


# 2.standardize column names so all tables use the same country-code and country-name columns:

In [10]:
coverage = coverage.rename(columns={"CODE": "ISO_3_CODE", "NAME": "COUNTRYNAME"})
incidence = incidence.rename(columns={"CODE": "ISO_3_CODE", "NAME": "COUNTRYNAME"})
cases = cases.rename(columns={"CODE": "ISO_3_CODE", "NAME": "COUNTRYNAME"})

# 3.drop fully blank/junk rows and fix YEAR type:

In [12]:
for name, df in [("coverage", coverage), ("incidence", incidence), ("cases", cases), ("intro", intro), ("schedule", schedule)]:
    n_missing = df["YEAR"].isna().sum()
    if n_missing > 0:
        print(f"{name}: dropping {n_missing} rows with missing YEAR")

coverage = coverage.dropna(subset=["YEAR"])
incidence = incidence.dropna(subset=["YEAR"])
cases = cases.dropna(subset=["YEAR"])
intro = intro.dropna(subset=["YEAR"])
schedule = schedule.dropna(subset=["YEAR"])

for df in [coverage, incidence, cases, intro, schedule]:
    df["YEAR"] = df["YEAR"].astype(int)

schedule: dropping 1 rows with missing YEAR


In [13]:
coverage = coverage.dropna(subset=["ISO_3_CODE"])
incidence = incidence.dropna(subset=["ISO_3_CODE"])
cases = cases.dropna(subset=["ISO_3_CODE"])
intro = intro.dropna(subset=["COUNTRYNAME"])
schedule = schedule.dropna(subset=["ISO_3_CODE"])

# YEAR should be a whole number, not float
for df in [coverage, incidence, cases, intro, schedule]:
    df["YEAR"] = df["YEAR"].astype(int)

# 4.for coverage's missing values:

In [14]:
# Just confirm counts after cleaning
print("Coverage rows with a usable COVERAGE %:", coverage["COVERAGE"].notna().sum())
print("Coverage rows with usable DOSES/TARGET_NUMBER:", coverage[["TARGET_NUMBER","DOSES"]].notna().all(axis=1).sum())

Coverage rows with a usable COVERAGE %: 230477
Coverage rows with usable DOSES/TARGET_NUMBER: 77317


# 5.strip whitespace from text columns:

In [15]:
for df in [coverage, incidence, cases, intro, schedule]:
    str_cols = df.select_dtypes(include="object").columns
    for col in str_cols:
        df[col] = df[col].str.strip()

C:\Users\avani\AppData\Local\Temp\ipykernel_5812\471464020.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  str_cols = df.select_dtypes(include="object").columns
C:\Users\avani\AppData\Local\Temp\ipykernel_5812\471464020.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migra

# 6.save the cleaned tables:

In [16]:
import os
os.makedirs("../data/cleaned", exist_ok=True)

coverage.to_csv("../data/cleaned/coverage_clean.csv", index=False)
incidence.to_csv("../data/cleaned/incidence_clean.csv", index=False)
cases.to_csv("../data/cleaned/cases_clean.csv", index=False)
intro.to_csv("../data/cleaned/intro_clean.csv", index=False)
schedule.to_csv("../data/cleaned/schedule_clean.csv", index=False)

print("Saved cleaned files.")

Saved cleaned files.
